# Convert TFLite model to PyTorch

This uses the model **face_detection_front.tflite** from [MediaPipe](https://github.com/google/mediapipe/tree/master/mediapipe/models).

Using conda environnement:
```
conda create -c pytorch -c conda-forge -n BlazeConv 'pytorch=1.6' jupyter opencv matplotlib
```
```
conda activate BlazeConv
```
```
pip install tflite
```

## Convert front camera TFLite model

In [ ]:
import os
import numpy as np
from collections import OrderedDict

### Get the weights from the TFLite file

Load the TFLite model using the FlatBuffers library:

In [ ]:
!wget -N https://github.com/google/mediapipe/raw/master/mediapipe/models/face_detection_front.tflite

In [ ]:
from tflite import Model

front_data = open("./face_detection_front.tflite", "rb").read()
front_model = Model.GetRootAsModel(front_data, 0)

In [ ]:
front_subgraph = front_model.Subgraphs(0)
front_subgraph.Name()

In [ ]:
def get_shape(tensor):
    return [tensor.Shape(i) for i in range(tensor.ShapeLength())]

List all the tensors in the graph:

In [ ]:
def print_graph(graph):
    for i in range(0, graph.TensorsLength()):
        tensor = graph.Tensors(i)
        print("%3d %30s %d %2d %s" % (i, tensor.Name(), tensor.Type(), tensor.Buffer(), 
                                      get_shape(graph.Tensors(i))))

print_graph(front_subgraph)

Make a look-up table that lets us get the tensor index based on the tensor name:

In [ ]:
front_tensor_dict = {(front_subgraph.Tensors(i).Name().decode("utf8")): i 
               for i in range(front_subgraph.TensorsLength())}

Grab only the tensors that represent weights and biases.

In [ ]:
def get_parameters(graph):
    parameters = {}
    for i in range(graph.TensorsLength()):
        tensor = graph.Tensors(i)
        if tensor.Buffer() > 0:
            name = tensor.Name().decode("utf8")
            parameters[name] = tensor.Buffer()
    return parameters

front_parameters = get_parameters(front_subgraph)
len(front_parameters)

The buffers are simply arrays of bytes. As the docs say,

> The data_buffer itself is an opaque container, with the assumption that the
> target device is little-endian. In addition, all builtin operators assume
> the memory is ordered such that if `shape` is [4, 3, 2], then index
> [i, j, k] maps to `data_buffer[i*3*2 + j*2 + k]`.

For weights and biases, we need to interpret every 4 bytes as being as float. On my machine, the native byte ordering is already little-endian so we don't need to do anything special for that.

Found some weights and biases stored as float16 instead of float32 corresponding to Type 1 instead of 0.

In [ ]:
def get_weights(model, graph, tensor_dict, tensor_name):
    i = tensor_dict[tensor_name]
    tensor = graph.Tensors(i)
    buffer = tensor.Buffer()
    shape = get_shape(tensor)
    assert(tensor.Type() == 0 or tensor.Type() == 1)  # FLOAT32
    
    W = model.Buffers(buffer).DataAsNumpy()
    if tensor.Type() == 0:
        W = W.view(dtype=np.float32)
    elif tensor.Type() == 1:
        W = W.view(dtype=np.float16)
    W = W.reshape(shape)
    return W

In [ ]:
W = get_weights(front_model, front_subgraph, front_tensor_dict, "conv2d/Kernel")
b = get_weights(front_model, front_subgraph, front_tensor_dict, "conv2d/Bias")
W.shape, b.shape

Now we can get the weights for all the layers and copy them into our PyTorch model.

### Convert the weights to PyTorch format

In [ ]:
import torch
from blazeface import BlazeFace

In [ ]:
front_net = BlazeFace()

In [ ]:
front_net

Make a lookup table that maps the layer names between the two models. We're going to assume here that the tensors will be in the same order in both models. If not, we should get an error because shapes don't match.

In [ ]:
def get_probable_names(graph):
    probable_names = []
    for i in range(0, graph.TensorsLength()):
        tensor = graph.Tensors(i)
        if tensor.Buffer() > 0 and (tensor.Type() == 0 or tensor.Type() == 1):
            probable_names.append(tensor.Name().decode("utf-8"))
    return probable_names

front_probable_names = get_probable_names(front_subgraph)
        
front_probable_names[:5]

In [ ]:
def get_convert(net, probable_names):
    convert = {}
    i = 0
    for name, params in net.state_dict().items():
        convert[name] = probable_names[i]
        i += 1
    return convert

front_convert = get_convert(front_net, front_probable_names)

Copy the weights into the layers.

Note that the ordering of the weights is different between PyTorch and TFLite, so we need to transpose them.

Convolution weights:

    TFLite:  (out_channels, kernel_height, kernel_width, in_channels)
    PyTorch: (out_channels, in_channels, kernel_height, kernel_width)

Depthwise convolution weights:

    TFLite:  (1, kernel_height, kernel_width, channels)
    PyTorch: (channels, 1, kernel_height, kernel_width)

In [ ]:
def build_state_dict(model, graph, tensor_dict, net, convert):
    new_state_dict = OrderedDict()

    for dst, src in convert.items():
        W = get_weights(model, graph, tensor_dict, src)
        print(dst, src, W.shape, net.state_dict()[dst].shape)

        if W.ndim == 4:
            if W.shape[0] == 1:
                W = W.transpose((3, 0, 1, 2))  # depthwise conv
            else:
                W = W.transpose((0, 3, 1, 2))  # regular conv
    
        new_state_dict[dst] = torch.from_numpy(W)
    return new_state_dict

front_state_dict = build_state_dict(front_model, front_subgraph, front_tensor_dict, front_net, front_convert)

In [ ]:
front_net.load_state_dict(front_state_dict, strict=True)

No errors? Then the conversion was successful!

### Save the checkpoint

In [ ]:
torch.save(front_net.state_dict(), "blazeface.pth")

## Convert back camera TFLite model

In [ ]:
!wget -N https://github.com/google/mediapipe/raw/master/mediapipe/models/face_detection_back.tflite

In [ ]:
back_data = open("./face_detection_back.tflite", "rb").read()
back_model = Model.GetRootAsModel(back_data, 0)
back_subgraph = back_model.Subgraphs(0)
back_subgraph.Name()

In [ ]:
print_graph(back_subgraph)

In [ ]:
back_tensor_dict = {(back_subgraph.Tensors(i).Name().decode("utf8")): i 
               for i in range(back_subgraph.TensorsLength())}

In [ ]:
back_parameters = get_parameters(back_subgraph)
len(back_parameters)

In [ ]:
W = get_weights(back_model, back_subgraph, back_tensor_dict, "conv2d/Kernel")
b = get_weights(back_model, back_subgraph, back_tensor_dict, "conv2d/Bias")
W.shape, b.shape

In [ ]:
back_net = BlazeFace(back_model=True)

In [ ]:
back_net

In [ ]:
back_probable_names = get_probable_names(back_subgraph)
back_probable_names[:5]

In [ ]:
back_convert = get_convert(back_net, back_probable_names)

In [ ]:
back_state_dict = build_state_dict(back_model, back_subgraph, back_tensor_dict, back_net, back_convert)

In [ ]:
back_net.load_state_dict(back_state_dict, strict=True)

In [ ]:
torch.save(back_net.state_dict(), "blazefaceback.pth")